# Paleomagnetic Poles

This notebook walks through the key steps involved in calculating a **paleomagnetic pole** from site-mean directions.

## From directions to poles

The inclination and declination of the geomagnetic field change with position on the globe. This makes it difficult to compare paleomagnetic results from different localities directly using directions. However, the position of the *magnetic pole* of a geocentric dipole is independent of observing locality. By converting directions to **virtual geomagnetic poles (VGPs)**, we can compare results from across a continent and can calculate a mean **paleomagnetic pole** for paleogeographic reconstruction.

## What we will do

1. **Convert site-mean directions to VGPs** using the dipole formula
2. **Calculate a mean paleomagnetic pole** using Fisher statistics on VGPs
3. **Test whether VGPs are Fisher-distributed** using a QQ plot
4. **Evaluate the pole** using the Deenen et al. (2011) test for adequate sampling of paleosecular variation

## Import libraries

In [ ]:
!pip install cartopy
!pip install pmagpy

In [ ]:
import pmagpy.pmag as pmag
import pmagpy.ipmag as ipmag
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import pandas as pd
import numpy as np

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## Osler Volcanic Group dataset

We use paleomagnetic data from the **Osler Volcanic Group**, a sequence of ca. 1108–1105 Ma basalt flows erupted during the Midcontinent Rift and exposed along the north shore of Lake Superior. The data combine reversed polarity site-mean directions from two studies archived in the MagIC database:

> Halls, H. (1974). A paleomagnetic reversal in the Osler Volcanic Group, northern Lake Superior. *Can. J. Earth Sci.*, 11, 1200–1207. doi:[10.1139/e74-113](https://doi.org/10.1139/e74-113)

> Swanson-Hysell, N. L., Vaughan, A. A., Mustain, M. R., and Asp, K. E. (2014). Confirmation of progressive plate motion during the Midcontinent Rift's early magmatic stage from the Osler Volcanic Group, Ontario, Canada. *Geochem. Geophys. Geosyst.*, 15, 2039–2047. doi:[10.1002/2013GC005180](https://doi.org/10.1002/2013GC005180)

We download the data from the MagIC database and use the **tilt-corrected** site-mean directions, following the approach of Tauxe et al. (2016). We combine the reversed polarity flows from Halls (1974) at Nipigon Strait with flows from the upper portion of the Simpson Island section from Swanson-Hysell et al. (2014), which are temporally equivalent. All directions are reversed polarity.

In [ ]:
# Download Halls (1974) data from MagIC
halls_dir = './data/Halls1974'
result, halls_file = ipmag.download_magic_from_id('20260', directory=halls_dir)
ipmag.unpack_magic(halls_file, dir_path=halls_dir, print_progress=False)

# Download Swanson-Hysell et al. (2014) data from MagIC
sh_dir = './data/SwansonHysell2014'
result, sh_file = ipmag.download_magic_from_id('18693', directory=sh_dir)
ipmag.unpack_magic(sh_file, dir_path=sh_dir, print_progress=False)

# Load sites tables
halls_sites = pd.read_csv(halls_dir + '/sites.txt', sep='\t', header=1)
sh_sites = pd.read_csv(sh_dir + '/sites.txt', sep='\t', header=1)

# Halls (1974): tilt-corrected reversed polarity flows from Nipigon Strait
halls_tc = halls_sites[(halls_sites.dir_tilt_correction == 100) &
                       (halls_sites.location.str.contains('Lower Reversed'))].copy()

# Swanson-Hysell et al. (2014): tilt-corrected upper section (>2082 m) from Simpson Island
sh_tc = sh_sites[(sh_sites.dir_tilt_correction == 100) &
                 (sh_sites.dir_dec.notna()) &
                 (sh_sites.height > 2082)].copy()

# Combine reversed polarity directions and per-site coordinates
reversed_dec = list(halls_tc.dir_dec.values) + list(sh_tc.dir_dec.values)
reversed_inc = list(halls_tc.dir_inc.values) + list(sh_tc.dir_inc.values)
site_lat = list(halls_tc.lat.values) + list(sh_tc.lat.values)
site_lon = list(halls_tc.lon.values) + list(sh_tc.lon.values)

print(f'Halls (1974) reversed sites: {len(halls_tc)}')
print(f'Swanson-Hysell et al. (2014) upper section sites: {len(sh_tc)}')
print(f'Total reversed polarity sites: {len(reversed_dec)}')

### Plot the site-mean directions

Let's look at the tilt-corrected reversed polarity directions on an equal-area stereonet. The reversed polarity directions (upper hemisphere) should cluster in the southeast, consistent with the expected Keweenawan reversed direction for Laurentia.

In [ ]:
plt.figure(figsize=(5, 5))
ipmag.plot_net()
ipmag.plot_di(dec=reversed_dec, inc=reversed_inc, color='red', markersize=40, label='Reversed')
plt.title('Tilt-corrected site-mean directions')
plt.legend(loc='upper right')
plt.show()

### Flip to a common polarity

Before calculating a paleomagnetic pole, all directions must be in the same polarity. Rather than manually labeling directions as normal or reversed, we use `pmag.flip()` which calculates the principal eigenvector of the directional distribution to bisect the data into two modes and flips the subordinate mode to match the principal mode. This approach is robust to mixed-polarity datasets. Since the combined Osler data are in reversed polarity, we then flip to the normal polarity convention (positive inclination) for VGP calculation.

In [ ]:
di_block = ipmag.make_di_block(reversed_dec, reversed_inc)
flipped = pmag.flip(di_block, combine=True)
normal_dec, normal_inc = ipmag.unpack_di_block(flipped)[:2]

In [ ]:
plt.figure(figsize=(5, 5))
ipmag.plot_net()
ipmag.plot_di(dec=normal_dec, inc=normal_inc, color='steelblue', markersize=40)
plt.title('All directions flipped to normal polarity')
plt.show()

## Step 1: Convert directions to Virtual Geomagnetic Poles (VGPs)

Each site-mean direction is converted to a VGP using the **dipole formula**. The procedure is:

1. Calculate the magnetic **colatitude** $p$ (angular distance from site to pole) from the inclination $I_m$ using the dipole equation:

$$p = \cot^{-1}\left(\frac{\tan I_m}{2}\right) = \tan^{-1}\left(\frac{2}{\tan I_m}\right)$$

2. Calculate the pole **latitude** $\lambda_p$:

$$\lambda_p = \sin^{-1}(\sin \lambda_s \cos p + \cos \lambda_s \sin p \cos D_m)$$

3. Calculate the pole **longitude** $\phi_p$ (with a sign test on $\beta$):

$$\beta = \sin^{-1}\left(\frac{\sin p \sin D_m}{\cos \lambda_p}\right)$$

The function `pmag.dia_vgp()` from PmagPy handles all of this. It also returns *dp* and *dm*, the semi-axes of the confidence ellipse about the pole (we pass $\alpha_{95}$ = 0 since we are just converting directions, not propagating confidence limits here).

In [ ]:
vgp_lon = []
vgp_lat = []

for i in range(len(normal_dec)):
    plon, plat, dp, dm = pmag.dia_vgp(normal_dec[i], normal_inc[i], 0, site_lat[i], site_lon[i])
    vgp_lon.append(plon)
    vgp_lat.append(plat)

# Display in a table
vgp_table = pd.DataFrame({
    'Site Lat (°N)': [round(v, 2) for v in site_lat],
    'Site Lon (°E)': [round(v, 2) for v in site_lon],
    'Dec (°)': normal_dec,
    'Inc (°)': normal_inc,
    'VGP Lon (°E)': [round(v, 1) for v in vgp_lon],
    'VGP Lat (°N)': [round(v, 1) for v in vgp_lat]
})
vgp_table.index.name = 'Site'
vgp_table

## Step 2: Calculate the mean paleomagnetic pole

We treat each VGP as a point on the unit sphere and calculate the Fisher mean. The procedure is the same as for directions, but we substitute VGP latitude for inclination and VGP longitude for declination.

By convention, statistics for VGP distributions use **upper-case** letters:
- $A_{95}$ = radius of the 95% confidence circle about the mean pole

In [ ]:
mean_pole = ipmag.fisher_mean(dec=vgp_lon, inc=vgp_lat)
ipmag.print_pole_mean(mean_pole)

In [ ]:
mean_pole_lon = mean_pole['dec']
mean_pole_lat = mean_pole['inc']
mean_pole_A95 = mean_pole['alpha95']

### Plot VGPs and mean pole on a map

In [ ]:
map_axis = ipmag.make_orthographic_map(central_longitude=mean_pole['dec'],
                                       central_latitude=mean_pole['inc'],
                                       figsize=(6, 6),
                                       land_edge_color=None)

# Plot individual VGPs
ipmag.plot_vgp(map_axis, vgp_lon=vgp_lon, vgp_lat=vgp_lat,
               color='steelblue', markersize=30, zorder=20, edge=None, alpha=0.75, 
               label='VGPs')

# Plot mean pole with A95 confidence circle
ipmag.plot_pole(map_axis, mean_pole_lon, mean_pole_lat, mean_pole_A95,
                marker='*', color='red', markersize=100, zorder=30, 
                label='Mean pole + $A_{95}$')

# Plot site locations
ipmag.plot_vgp(map_axis, vgp_lon=site_lon, vgp_lat=site_lat,
               color='green', marker='s', markersize=60, zorder=20, 
               label='Site locations')

plt.legend(loc='lower center', fontsize=11)

plt.title('VGPs and Mean Paleomagnetic Pole')
plt.show()

### Calculate the paleolatitude

The angular distance between the site and the mean pole gives the magnetic colatitude. Subtracting from 90° yields the paleolatitude of the site at the time of magnetization.

First let's get the mean site longitude/latitude. We can use Fisher statistics for this.

In [ ]:
mean_site_location = ipmag.fisher_mean(dec=site_lon, inc=site_lat)
mean_site_lon = mean_site_location['dec']
mean_site_lat = mean_site_location['inc']
print(f'Mean site location: {mean_site_lat:.1f}°N, {mean_site_lon:.1f}°E')

In [ ]:
colatitude = pmag.angle([mean_pole_lon, mean_pole_lat], [mean_site_lon, mean_site_lat])
paleolatitude = 90 - colatitude[0]
print(f'Paleolatitude: {paleolatitude:.1f}°N')

## Step 3: Fisher QQ test

The Fisher distribution is the spherical analog of the normal distribution. If VGPs are drawn from a Fisher distribution, it supports the interpretation that we are sampling a single population of directions scattered by geomagnetic secular variation.

The **QQ plot** compares the observed distribution of VGPs against what would be expected from a Fisher distribution:
- **Uniform plot** (left): tests whether longitudes are uniformly distributed (as expected for a Fisher distribution)
- **Exponential plot** (right): tests whether colatitudes follow an exponential distribution

Test statistics $M_u$ and $M_e$ are compared against critical values. If either exceeds its critical value, the Fisher hypothesis is rejected.

In [ ]:
fishqq_results = ipmag.fishqq(lon=vgp_lon, lat=vgp_lat)
plt.show()

print(f"\nMu = {fishqq_results['Mu']:.3f}  (critical = {fishqq_results['Mu_critical']:.3f})")
print(f"Me = {fishqq_results['Me']:.3f}  (critical = {fishqq_results['Me_critical']:.3f})")
print(f"\nResult: {fishqq_results['Test_result']}")

If the data fall along the diagonal line in both plots and $M_u < M_{u,critical}$ and $M_e < M_{e,critical}$, the VGP distribution is consistent with being drawn from a Fisher distribution.

## Step 4: Deenen et al. (2011) test

A key question in paleomagnetic pole determination is whether the data have adequately sampled **paleosecular variation (PSV)**. If sites were magnetized over too short a time interval, VGPs will be too tightly clustered — the pole may be precise but not accurate.

Deenen et al. (2011) established empirical bounds on $A_{95}$ as a function of the number of sites $N$:

$$A_{95,\text{min}} = 12 \cdot N^{-0.40}$$
$$A_{95,\text{max}} = 82 \cdot N^{-0.63}$$

- If $A_{95} < A_{95,\text{min}}$: VGPs are **too tightly clustered** — PSV has probably *not* been adequately sampled (e.g., flows erupted in rapid succession)
- If $A_{95} > A_{95,\text{max}}$: VGPs are **too dispersed** — there may be additional sources of scatter beyond PSV (e.g., tectonic disturbance, poor isolation of ChRM)
- If $A_{95,\text{min}} \leq A_{95} \leq A_{95,\text{max}}$: the dispersion is **consistent with adequate sampling of PSV**

In [ ]:
def deenen_test(N, A95):
    """Evaluate A95 against Deenen et al. (2011) criteria."""
    A95_min = 12 * N**(-0.40)
    A95_max = 82 * N**(-0.63)

    print(f'N = {N}')
    print(f'A95 = {A95:.1f}°')
    print(f'A95_min = {A95_min:.1f}°')
    print(f'A95_max = {A95_max:.1f}°')
    print()

    if A95 < A95_min:
        print(f'FAIL: A95 ({A95:.1f}°) < A95_min ({A95_min:.1f}°)')
        print('VGPs are too tightly clustered — PSV not adequately sampled.')
    elif A95 > A95_max:
        print(f'FAIL: A95 ({A95:.1f}°) > A95_max ({A95_max:.1f}°)')
        print('VGPs are too dispersed — additional scatter beyond PSV.')
    else:
        print(f'PASS: A95_min ({A95_min:.1f}°) <= A95 ({A95:.1f}°) <= A95_max ({A95_max:.1f}°)')
        print('Dispersion is consistent with adequate sampling of PSV.')

In [ ]:
deenen_test(mean_pole['n'], mean_pole['alpha95'])

### Visualize the Deenen bounds

Let's plot the $A_{95}$ envelope as a function of $N$ and see where our data fall.

In [ ]:
N_range = np.arange(5, 100)
A95_min_curve = 12 * N_range**(-0.40)
A95_max_curve = 82 * N_range**(-0.63)

plt.figure(figsize=(7, 4))
plt.fill_between(N_range, A95_min_curve, A95_max_curve, alpha=0.2, color='green', label='Acceptable range')
plt.plot(N_range, A95_min_curve, 'g--', linewidth=1, label='$A_{95,min}$')
plt.plot(N_range, A95_max_curve, 'g--', linewidth=1, label='$A_{95,max}$')
plt.plot(mean_pole['n'], mean_pole['alpha95'], 'r*', markersize=15, zorder=5, label='Our pole')
plt.xlabel('Number of sites (N)', fontsize=12)
plt.ylabel('$A_{95}$ (°)', fontsize=12)
plt.title('Deenen et al. (2011) test', fontsize=13)
plt.legend(fontsize=10)
plt.xlim(5, 80)
plt.ylim(0, 25)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary: Criteria for a reliable paleomagnetic pole

As you will read in Chapter 7, there are numerous aspects of the data associated with a paleomagnetic pole that we can consider when assessing its reliability:

1. **Field tests of paleomagnetic stability** (reversals test, fold test, conglomerate test) provide crucial information about the timing and primary nature of the ChRM.

3. **Number of VGPs $\geq$ 10** is required for reasonable averaging of secular variation and for meaningful estimation of angular dispersion.

4. **Dispersion of VGPs should be consistent** with adequate sampling of geomagnetic secular variation — the Deenen test provides a quantitative check. The distinction between a *precise* pole ($A_{95}$ is small) and an *accurate* pole (actually represents the time-averaged field) is important. At the same time, disperson that is too large can indicate a poor record.

5. **Age control on the pole** is essential for the paleomagnetic pole to be useful. There need to be high-quality age constraints that are well-documented for the pole to be used for paleogeographic reconstruction.